# Bab 7: Membangun Aplikasi Obrolan
## Mulai Cepat Azure OpenAI API


## Ikhtisar
Notebook ini diadaptasi dari [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) yang juga mencakup notebook yang mengakses layanan [OpenAI](notebook-openai.ipynb).

API Python OpenAI juga bekerja dengan Azure OpenAI, dengan beberapa modifikasi. Pelajari lebih lanjut tentang perbedaannya di sini: [Cara beralih antara endpoint OpenAI dan Azure OpenAI dengan Python](https://learn.microsoft.com/azure/ai-services/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)

Untuk contoh memulai cepat lainnya, silakan merujuk ke [Dokumentasi Memulai Cepat Azure OpenAI](https://learn.microsoft.com/azure/cognitive-services/openai/quickstart?pivots=programming-language-studio&WT.mc_id=academic-105485-koreyst) resmi


## Daftar Isi  

[Ikhtisar](#overview)  
[Memulai dengan Azure OpenAI Service](#getting-started-with-azure-openai-service)  
[Membangun prompt pertama Anda](#build-your-first-prompt)  

[Kasus Penggunaan](#use-cases)    
[1. Meringkas Teks](#summarize-text)  
[2. Mengklasifikasikan Teks](#classify-text)  
[3. Menghasilkan Nama Produk Baru](#generate-new-product-names)  
[4. Melatih Ulang Klasifikator](#fine-tune-a-classifier)  
[5. Embeddings](#embeddings)

[Referensi](#references)


### Memulai dengan Azure OpenAI Service

Pelanggan baru perlu [mengajukan permohonan akses](https://aka.ms/oai/access?WT.mc_id=academic-105485-koreyst) ke Azure OpenAI Service.  
Setelah persetujuan selesai, pelanggan dapat masuk ke portal Azure, membuat sumber daya Azure OpenAI Service, dan mulai bereksperimen dengan model melalui studio  

[Sumber daya hebat untuk memulai dengan cepat](https://techcommunity.microsoft.com/blog/educatordeveloperblog/azure-openai-service-is-now-generally-available/3719177?WT.mc_id=academic-105485-koreyst)


### Bangun prompt pertama Anda  
Latihan singkat ini akan memberikan pengenalan dasar untuk mengirimkan prompt ke model OpenAI untuk tugas sederhana "ringkasan".


**Langkah-langkah**:  
1. Instal perpustakaan OpenAI di lingkungan python Anda  
2. Muat perpustakaan pembantu standar dan atur kredensial keamanan OpenAI khas Anda untuk Layanan OpenAI yang telah Anda buat  
3. Pilih model untuk tugas Anda  
4. Buat prompt sederhana untuk model  
5. Kirim permintaan Anda ke API model!


### 1. Instal OpenAI


  > [!NOTE] Langkah ini tidak diperlukan jika menjalankan notebook ini di Codespaces atau di dalam Devcontainer


In [ ]:
%pip install openai python-dotenv

### 2. Impor perpustakaan pembantu dan instansiasi kredensial


In [ ]:
import os
from openai import OpenAI
import numpy as np
from dotenv import load_dotenv
load_dotenv()

#validate data inside .env file

# The Responses API is served from the Azure OpenAI (Microsoft Foundry) v1 endpoint,
# so we point the OpenAI client at <your-endpoint>/openai/v1/ (no api_version needed).
endpoint = os.environ['AZURE_OPENAI_ENDPOINT']
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{endpoint.rstrip('/')}/openai/v1/",
  )


### 3. Menemukan model yang tepat  
Model seperti GPT-4o dan GPT-4o mini dapat memahami dan menghasilkan bahasa alami. Azure OpenAI di Microsoft Foundry menawarkan berbagai kemampuan model, masing-masing dengan tingkat kekuatan dan kecepatan yang berbeda sesuai untuk berbagai tugas. 

[Azure OpenAI in Microsoft Foundry models](https://learn.microsoft.com/en-us/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)



In [ ]:
# Select the deployment name configured in your .env file
model = os.environ['AZURE_OPENAI_DEPLOYMENT']


## 4. Desain Prompt  

"Keajaiban model bahasa besar adalah dengan dilatih untuk meminimalkan kesalahan prediksi ini di atas banyak sekali teks, model akhirnya belajar konsep yang berguna untuk prediksi ini. Misalnya, mereka belajar konsep seperti"(1):

* bagaimana mengeja
* bagaimana tata bahasa bekerja
* bagaimana memparafrasekan
* bagaimana menjawab pertanyaan
* bagaimana melakukan percakapan
* bagaimana menulis dalam banyak bahasa
* bagaimana melakukan pemrograman
* dll.

#### Cara mengendalikan model bahasa besar  
"Dari semua input ke model bahasa besar, yang paling berpengaruh adalah prompt teks(1).

Model bahasa besar dapat diprompt untuk menghasilkan output dengan beberapa cara:

- Instruksi: Beritahu model apa yang Anda inginkan
- Penyelesaian: Mendorong model untuk menyelesaikan awal dari apa yang Anda inginkan
- Demonstrasi: Tunjukkan ke model apa yang Anda inginkan, dengan cara:
- Beberapa contoh dalam prompt
- Ratusan atau ribuan contoh dalam dataset pelatihan fine-tuning



#### Ada tiga pedoman dasar dalam membuat prompt:

**Tunjukkan dan jelaskan**. Jelaskan apa yang Anda ingin dengan instruksi, contoh, atau kombinasi keduanya. Jika Anda ingin model mengurutkan daftar item secara alfabetis atau mengklasifikasikan paragraf berdasarkan sentimen, tunjukkan bahwa itu yang Anda inginkan.

**Sediakan data berkualitas**. Jika Anda mencoba membuat pengklasifikasi atau membuat model mengikuti pola, pastikan ada cukup contoh. Pastikan untuk memeriksa contoh Anda — model biasanya cukup pintar untuk melihat kesalahan ejaan dasar dan memberi Anda respons, tetapi juga mungkin menganggap ini disengaja dan dapat memengaruhi respons.

**Periksa pengaturan Anda.** Pengaturan temperature dan top_p mengontrol seberapa deterministik model dalam menghasilkan respons. Jika Anda meminta respons dengan hanya satu jawaban benar, Anda ingin mengatur keduanya lebih rendah. Jika Anda menginginkan respons yang lebih beragam, Anda bisa mengatur keduanya lebih tinggi. Kesalahan nomor satu yang sering orang lakukan dengan pengaturan ini adalah menganggap bahwa ini adalah kontrol "kecerdasan" atau "kreativitas".


Sumber: https://learn.microsoft.com/azure/ai-services/openai/overview


image sedang membuat prompt teks pertamamu!


### 5. Kirim!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Ulangi panggilan yang sama, bagaimana perbandingan hasilnya?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Ringkas Teks  
#### Tantangan  
Ringkas teks dengan menambahkan 'tl;dr:' di akhir sebuah bagian teks. Perhatikan bagaimana model memahami cara melakukan sejumlah tugas tanpa instruksi tambahan. Anda dapat bereksperimen dengan prompt yang lebih deskriptif daripada tl;dr untuk memodifikasi perilaku model dan menyesuaikan ringkasan yang Anda terima(3).  

Pekerjaan terkini telah menunjukkan peningkatan substansial pada banyak tugas dan tolok ukur NLP dengan melakukan pra-pelatihan pada korpus teks besar diikuti oleh penyetelan ulang pada tugas tertentu. Walaupun biasanya arsitekturnya tidak tergantung pada tugas, metode ini masih memerlukan dataset penyetelan ulang spesifik tugas yang terdiri dari ribuan atau puluhan ribu contoh. Sebaliknya, manusia umumnya dapat melakukan tugas bahasa baru hanya dari beberapa contoh atau instruksi sederhana - sesuatu yang masih sangat sulit dilakukan oleh sistem NLP saat ini. Di sini kami menunjukkan bahwa memperbesar model bahasa sangat meningkatkan kinerja beberapa tembakan (few-shot) yang tidak tergantung pada tugas, terkadang bahkan mencapai kompetisi dengan pendekatan penyetelan ulang mutakhir sebelumnya.  



Tl;dr  


# Latihan untuk beberapa kasus penggunaan  
1. Meringkas Teks  
2. Mengklasifikasikan Teks  
3. Menghasilkan Nama Produk Baru
4. Embeddings
5. Melatih ulang sebuah pengklasifikasi


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\ntl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klasifikasikan Teks  
#### Tantangan  
Klasifikasikan item ke dalam kategori yang diberikan pada saat inferensi. Dalam contoh berikut, kami memberikan kategori dan teks untuk diklasifikasikan dalam prompt (*playground_reference). 

Pertanyaan Pelanggan: Halo, salah satu tombol pada keyboard laptop saya baru-baru ini rusak dan saya butuh penggantinya:

Kategori yang diklasifikasikan:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Buat Nama Produk Baru
#### Tantangan
Buat nama produk dari kata-kata contoh. Di sini kami memasukkan dalam prompt informasi tentang produk yang akan kami buat namanya. Kami juga menyediakan contoh yang serupa untuk menunjukkan pola yang kami harapkan. Kami juga telah mengatur nilai temperatur tinggi untuk meningkatkan keacakan dan respons yang lebih inovatif.

Deskripsi produk: Pembuat milkshake rumahan
Kata kunci: cepat, sehat, ringkas.
Nama produk: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Deskripsi produk: Sepasang sepatu yang dapat pas untuk ukuran kaki mana pun.
Kata kunci: adaptif, pas, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


## Embeddings
Bagian ini akan menunjukkan cara mengambil embeddings, dan menemukan kesamaan antara kata, kalimat, dan dokumen. Untuk menjalankan notebook berikut Anda perlu men-deploy model yang menggunakan `text-embedding-ada-002` sebagai model dasar dan mengatur nama deployment-nya di dalam file .env, menggunakan variabel `AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT`.


### Taksonomi Model - Memilih model embedding

**Taksonomi model**: {family} - {capability} - {input-type} - {identifier}  

{family}     --> text-embedding  (keluarga embeddings)  
{capability} --> ada             (semua model embedding lain akan dihentikan pada 2024)  
{input-type} --> n/a             (hanya ditentukan untuk model pencarian)  
{identifier} --> 002             (versi 002)  

model = 'text-embedding-ada-002'


  > [!NOTE] Langkah berikut tidak diperlukan jika menjalankan notebook ini di Codespaces atau di dalam Devcontainer


In [ ]:
# Dependencies for embeddings_utils
%pip install matplotlib plotly scikit-learn pandas

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
text = 'the quick brown fox jumped over the lazy dog'
model= os.environ['AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT']
client.embeddings.create(input='[text]', model=model).data[0].embedding

In [ ]:

# compare several words
automobile_embedding  = client.embeddings.create(input='automobile', model=model).data[0].embedding
vehicle_embedding     = client.embeddings.create(input='vehicle', model=model).data[0].embedding
dinosaur_embedding    = client.embeddings.create(input='dinosaur', model=model).data[0].embedding
stick_embedding       = client.embeddings.create(input='stick', model=model).data[0].embedding

print(cosine_similarity(automobile_embedding, vehicle_embedding))
print(cosine_similarity(automobile_embedding, dinosaur_embedding))
print(cosine_similarity(automobile_embedding, stick_embedding))

## Membandingkan artikel dari dataset berita harian cnn
source: https://huggingface.co/datasets/cnn_dailymail


In [ ]:
import pandas as pd
cnn_daily_articles = ['BREMEN, Germany -- Carlos Alberto, who scored in FC Porto\'s Champions League final victory against Monaco in 2004, has joined Bundesliga club Werder Bremen for a club record fee of 7.8 million euros ($10.7 million). Carlos Alberto enjoyed success at FC Porto under Jose Mourinho. "I\'m here to win titles with Werder," the 22-year-old said after his first training session with his new club. "I like Bremen and would only have wanted to come here." Carlos Alberto started his career with Fluminense, and helped them to lift the Campeonato Carioca in 2002. In January 2004 he moved on to FC Porto, who were coached by José Mourinho, and the club won the Portuguese title as well as the Champions League. Early in 2005, he moved to Corinthians, where he impressed as they won the Brasileirão,but in 2006 Corinthians had a poor season and Carlos Alberto found himself at odds with manager, Emerson Leão. Their poor relationship came to a climax at a Copa Sul-Americana game against Club Atlético Lanús, and Carlos Alberto declared that he would not play for Corinthians again while Leão remained as manager. Since January this year he has been on loan with his first club Fluminense. Bundesliga champions VfB Stuttgart said on Sunday that they would sign a loan agreement with Real Zaragoza on Monday for Ewerthon, the third top Brazilian player to join the German league in three days. A VfB spokesman said Ewerthon, who played in the Bundesliga for Borussia Dortmund from 2001 to 2005, was expected to join the club for their pre-season training in Austria on Monday. On Friday, Ailton returned to Germany where he was the league\'s top scorer in 2004, signing a one-year deal with Duisburg on a transfer from Red Star Belgrade. E-mail to a friend .',
                        '(CNN) -- Football superstar, celebrity, fashion icon, multimillion-dollar heartthrob. Now, David Beckham is headed for the Hollywood Hills as he takes his game to U.S. Major League Soccer. CNN looks at how Bekham fulfilled his dream of playing for Manchester United, and his time playing for England. The world\'s famous footballer has begun a five-year contract with the Los Angeles Galaxy team, and on Friday Beckham will meet the press and reveal his new shirt number. This week, we take an in depth look at the life and times of Beckham, as CNN\'s very own "Becks," Becky Anderson, sets out to examine what makes the man tick -- as footballer, fashion icon and global phenomenon. It\'s a long way from the streets of east London to the Hollywood Hills and Becky charts Beckham\'s incredible rise to football stardom, a journey that has seen his skills grace the greatest stages in world soccer. She goes in pursuit of the current hottest property on the sports/celebrity circuit in the U.S. and along the way explores exactly what\'s behind the man with the golden boot. CNN will look back at the life of Beckham, the wonderfully talented youngster who fulfilled his dream of playing for Manchester United, his marriage to pop star Victoria, and the trials and tribulations of playing for England. We\'ll look at the highs (scoring against Greece), the lows (being sent off during the World Cup), the Man. U departure for the Galacticos of Madrid -- and now the Home Depot stadium in L.A. We\'ll ask how Beckham and his family will adapt to life in Los Angeles -- the people, the places to see and be seen and the celebrity endorsement. Beckham is no stranger to exposure. He has teamed with Reggie Bush in an Adidas commercial, is the face of Motorola, is the face on a PlayStation game and doesn\'t need fashion tips as he has his own international clothing line. But what does the star couple need to do to become an accepted part of Tinseltown\'s glitterati? The road to major league football in the U.S.A. is a well-worn route for some of the world\'s greatest players. We talk to some of the former greats who came before him and examine what impact these overseas stars had on U.S. soccer and look at what is different now. We also get a rare glimpse inside the David Beckham academy in L.A, find out what drives the kids and who are their heroes. The perception that in the U.S.A. soccer is a "game for girls" after the teenage years is changing. More and more young kids are choosing the European game over the traditional U.S. sports. E-mail to a friend .',
                        'LOS ANGELES, California (CNN) -- Youssif, the 5-year-old burned Iraqi boy, rounded the corner at Universal Studios when suddenly the little boy hero met his favorite superhero. Youssif has always been a huge Spider-Man fan. Meeting him was "my favorite thing," he said. Spider-Man was right smack dab in front of him, riding a four-wheeler amid a convoy of other superheroes. The legendary climber of buildings and fighter of evil dismounted, walked over to Youssif and introduced himself. Spidey then gave the boy from a far-away land a gentle hug, embracing him in his iconic blue and red tights. He showed Youssif a few tricks, like how to shoot a web from his wrist. Only this time, no web was spun. "All right Youssif!" Spider-Man said after the boy mimicked his wrist movement. Other superheroes crowded around to get a closer look. Even the Green Goblin stopped his villainous ways to tell the boy hi. Youssif remained unfazed. He didn\'t take a liking to Spider-Man\'s nemesis. Spidey was just too cool. "It was my favorite thing," the boy said later. "I want to see him again." He then felt compelled to add: "I know it\'s not the real Spider-Man." This was the day of dreams when the boy\'s nightmares were, at least temporarily, forgotten. He met SpongeBob, Lassie and a 3-year-old orangutan named Archie. The hairy, brownish-red primate took to the boy, grabbing his hand and holding it. Even when Youssif pulled away, Archie would inch his hand back toward the boy\'s and then snatch it. See Youssif enjoy being a boy again » . The boy giggled inside a play area where sponge-like balls shot out of toy guns. It was a far different artillery than what he was used to seeing in central Baghdad, as recently as a week ago. He squealed with delight and raced around the room collecting as many balls as he could. He rode a tram through the back stages at Universal Studios. At one point, the car shook. Fire and smoke filled the air, debris cascaded down and a big rig skidded toward the vehicle. The boy and his family survived the pretend earthquake unscathed. "Even I was scared," the dad said. "Well, I wasn\'t," Youssif replied. The father and mother grinned from ear to ear throughout the day. Youssif pushed his 14-month-old sister, Ayaa, in a stroller. "Did you even need to ask us if we were interested in coming here?" Youssif\'s father said in amazement. "Other than my wedding day, this is the happiest day of my life," he said. Just a day earlier, the mother and father talked about their journey out of Iraq and to the United States. They also discussed that day nine months ago when masked men grabbed their son outside the family home, doused him in gas and set him on fire. His mother heard her boy screaming from inside. The father sought help for his boy across Baghdad, but no one listened. He remembers his son\'s two months of hospitalization. The doctors didn\'t use anesthetics. He could hear his boy\'s piercing screams from the other side of the hospital. Watch Youssif meet his doctor and play with his little sister » . The father knew that speaking to CNN would put his family\'s lives in jeopardy. The possibility of being killed was better than seeing his son suffer, he said. "Anything for Youssif," he said. "We had to do it." They described a life of utter chaos in Baghdad. Neighbors had recently given birth to a baby girl. Shortly afterward, the father was kidnapped and killed. Then, there was the time when some girls wore tanktops and jeans. They were snatched off the street by gunmen. The stories can be even more gruesome. The couple said they had heard reports that a young girl was kidnapped and beheaded --and her killers sewed a dog\'s head on the corpse and delivered it to her family\'s doorstep. "These are just some of the stories," said Youssif\'s mother, Zainab. Under Saddam Hussein, there was more security and stability, they said. There was running water and electricity most of the time. But still life was tough under the dictator, like the time when Zainab\'s uncle disappeared and was never heard from again after he read a "religious book," she said. Sitting in the parking lot of a Target in suburban Los Angeles, Youssif\'s father watched as husbands and wives, boyfriends and girlfriends, parents and their children, came and went. Some held hands. Others smiled and laughed. "Iraq finished," he said in what few English words he knows. He elaborated in Arabic: His homeland won\'t be enjoying such freedoms anytime soon. It\'s just not possible. Too much violence. Too many killings. His two children have only seen war. But this week, the family has seen a much different side of America -- an outpouring of generosity and a peaceful nation at home. "It\'s been a dream," the father said. He used to do a lot of volunteer work back in Baghdad. "Maybe that\'s why I\'m being helped now," the father said. At Universal Studios, he looked out across the valley below. The sun glistened off treetops and buildings. It was a picturesque sight fit for a Hollywood movie. "Good America, good America," he said in English. E-mail to a friend . CNN\'s Arwa Damon contributed to this report.'
]

cnn_daily_article_highlights = ['Werder Bremen pay a club record $10.7 million for Carlos Alberto .\nThe Brazilian midfielder won the Champions League with FC Porto in 2004 .\nSince January he has been on loan with his first club, Fluminense .',
                                'Beckham has agreed to a five-year contract with Los Angeles Galaxy .\nNew contract took effect July 1, 2007 .\nFormer English captain to meet press, unveil new shirt number Friday .\nCNN to look at Beckham as footballer, fashion icon and global phenomenon .',
                                'Boy on meeting Spider-Man: "It was my favorite thing"\nYoussif also met SpongeBob, Lassie and an orangutan at Universal Studios .\nDad: "Other than my wedding day, this is the happiest day of my life"'
]

cnn_df = pd.DataFrame({"articles":cnn_daily_articles, "highligths":cnn_daily_article_highlights})

cnn_df.head()

In [ ]:
article1_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[0], model=model).data[0].embedding
article2_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[1], model=model).data[0].embedding
article3_embedding    = client.embeddings.create(input=cnn_df.articles.iloc[2], model=model).data[0].embedding

print(cosine_similarity(article1_embedding, article2_embedding))
print(cosine_similarity(article1_embedding, article3_embedding))

# Referensi  
- [Microsoft Foundry - Model Azure OpenAI](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst)  
- [Portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  


# Untuk Bantuan Lebih Lanjut  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com)


# Kontributor
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
